# Political Discourse Analysis on Social Media
## Capstone Project 1 — Data Collection & Analysis
### Course: Introduction to Data, Knowledge and Ontology
**Student:** Arsenii Linkin  
**Dataset:** Telegram (15,000 posts) + Twitter/X (14,607 posts)  
**Tools:** Telethon, Apify, VADER, Plotly, pandas

## Setup

In [ ]:
import subprocess, sys

libs = ["pandas","numpy","plotly","vaderSentiment","tqdm"]
for lib in libs:
    subprocess.check_call([sys.executable,"-m","pip","install",lib,"-q"])

print("all libraries ready")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ── load clean datasets ─────────────────────────────────────────────────────
tg = pd.read_csv("data/clean/telegram_raw_clean.csv", low_memory=False)
tw = pd.read_csv("data/clean/twitter_raw_clean.csv", low_memory=False)

tg["date"] = pd.to_datetime(tg["date"], utc=True, errors="coerce")
tw["date"] = pd.to_datetime(tw["date"], utc=True, errors="coerce")

both = pd.concat([tg, tw], ignore_index=True)

print(f"Telegram : {len(tg):,} rows  |  {tg.shape[1]} columns")
print(f"Twitter  : {len(tw):,} rows  |  {tw.shape[1]} columns")
print(f"Combined : {len(both):,} rows")
print(f"Date range Telegram : {tg['date'].min().date()} -> {tg['date'].max().date()}")
print(f"Date range Twitter  : {tw['date'].min().date()} -> {tw['date'].max().date()}")

## Step 1 — Case Study Selection

Two case studies drive this project. They are different enough to support cross-platform comparison, but share the same underlying theme — how geopolitical conflict is discussed online.

**Case Study 1: Narrative Framing on Telegram**  
Four major Telegram channels (Reuters, BBC, NYT, Disclose.tv) are tracked to measure how Western media frames ongoing conflicts. The central question is whether channels with different editorial stances produce measurably different sentiment patterns around the same leaders and hotspots.

**Case Study 2: Real-Time Discourse on Twitter/X**  
A keyword-filtered Twitter sample captures public reaction to the same set of geopolitical actors and events. The goal is to compare the *tone* of informal public discourse against the *tone* of structured media output on Telegram.

**What are we comparing?**  
- Sentiment scores across platforms for the same leaders and hotspots  
- Mention frequency (who/what gets covered most)  
- Engagement patterns (views, forwards, reactions on Telegram; likes, retweets on Twitter)  
- Narrative framing differences between curated media channels and raw public discourse

## Step 2 — Data Source Identification

| Source | Method | Tool | Used |
|--------|--------|------|------|
| Telegram public channels | MTProto API client | Telethon | Yes — Dataset 1 |
| Twitter/X keyword search | API-mediated scraping | Apify actor | Yes — Dataset 2 |
| GDELT Global Knowledge Graph | REST API (free) | requests | Evaluated, not used |

**Source 1 — Telegram via Telethon:** Provides full message history from public channels including text, engagement counts, media flags, and channel subscriber data. The only viable method for Telegram channel archives.

**Source 2 — Twitter/X via Apify:** Official API limits made direct access impractical for research-scale collection. Apify's maintained actor handles proxy rotation and returns structured JSON matching this project's schema.

**Source 3 — GDELT (evaluated):** Provides event-based news data globally, but the event-centric format (actor-action-target triples) does not map naturally to the post-based schema used here. Noted for future enrichment work.

## Step 3 — Data Collection Strategy

**Method:** One-time batch collection. No live polling — the goal is a historical cross-section, not a monitoring system.

**Pipeline:**
1. Telethon authenticates via MTProto (API ID + phone in `.env`) and iterates Telegram channel history
2. Apify actor runs against keyword list from `config/keywords.yaml`, returns structured JSON
3. Both scrapers run VADER sentiment enrichment + entity matching inline at collection time
4. Results are saved to `data/raw/` as CSV, then cleaned to `data/clean/`

**Storage:** CSV (primary), JSON (backup). CSV chosen for pandas compatibility and portability.

**Tools:** Telethon, apify-client, vaderSentiment, pandas, PyYAML, python-dotenv, Typer, tqdm

## Step 4 — Data Collection

In [ ]:
# ── raw dataset stats ──────────────────────────────────────────────────────
print("=" * 60)
print("DATA COLLECTION SUMMARY")
print("=" * 60)

for name, df in [("Telegram", tg), ("Twitter", tw)]:
    print(f"\n{name} Dataset:")
    print(f"  Records   : {len(df):,}")
    print(f"  Attributes: {df.shape[1]}")
    print(f"  Columns   : {list(df.columns)}")

print(f"\nTotal records : {len(tg)+len(tw):,}  (minimum required: 10,000)")
print(f"Total attrs   : {tg.shape[1]+tw.shape[1]}  (minimum required: 15)")

print("\nTelegram channels:")
print(tg.groupby("channel_username")[["channel_name","channel_subscribers","views"]].agg({
    "channel_name":"first","channel_subscribers":"first","views":"sum"
}).to_string())

## Step 5 — Data Cleaning

In [ ]:
import pandas as pd

tg_raw = pd.read_csv("data/raw/telegram_raw.csv", low_memory=False)
tw_raw = pd.read_csv("data/raw/twitter_raw.csv", low_memory=False)

print("=" * 60)
print("CLEANING REPORT")
print("=" * 60)

for name, raw, clean in [("Telegram", tg_raw, tg), ("Twitter", tw_raw, tw)]:
    removed = len(raw) - len(clean)
    null_before = raw.isnull().sum().sum()
    null_after  = clean.isnull().sum().sum()
    dupes       = raw.duplicated().sum()
    print(f"\n{name} Dataset:")
    print(f"  Rows before cleaning : {len(raw):,}")
    print(f"  Rows after cleaning  : {len(clean):,}")
    print(f"  Rows removed         : {removed}")
    print(f"  Nulls before         : {null_before}")
    print(f"  Nulls after          : {null_after}")
    print(f"  Duplicates in raw    : {dupes}")
    print(f"  Added columns        : source_type (provenance tag)")

print("\nCleaning steps applied to BOTH datasets:")
print("  1. Drop rows with empty or null text")
print("  2. Deduplicate by message/tweet id + channel")
print("  3. Parse and normalize datetime to ISO-8601 UTC")
print("  4. Fill numeric engagement nulls with 0")
print("  5. Cast engagement columns to int")
print("  6. Add source_type = 'api_scrape' column")

## Visualizations

All charts are interactive — hover, zoom, click legend items to filter.  
Built with Plotly. Data source: `data/clean/telegram_raw_clean.csv` + `data/clean/twitter_raw_clean.csv`

### Chart 1 — Sentiment Distribution: Telegram vs Twitter

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type":"pie"},{"type":"pie"}]],
    subplot_titles=["Telegram (15,000 posts)","Twitter (14,607 posts)"]
)

COLORS = {"positive":"#27ae60","negative":"#e74c3c","neutral":"#f39c12"}

for i, (df, name) in enumerate([(tg,"Telegram"),(tw,"Twitter")],1):
    counts = df["sentiment_label"].value_counts()
    fig.add_trace(go.Pie(
        labels=counts.index,
        values=counts.values,
        name=name,
        marker_colors=[COLORS.get(l,"#95a5a6") for l in counts.index],
        textinfo="label+percent",
        hovertemplate="%{label}: %{value:,} posts (%{percent})<extra></extra>"
    ), row=1, col=i)

fig.update_layout(
    title=dict(text="Sentiment Distribution by Platform", font=dict(size=18)),
    height=420,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.1)
)
fig.show()

### Chart 2 — Most Mentioned Political Leaders (Both Platforms)

In [ ]:
def parse_entities(series):
    counts = Counter()
    for val in series.dropna():
        for e in str(val).split(","):
            e = e.strip()
            if e and e not in ("nan","None",""):
                counts[e] += 1
    return counts

leader_counts_tg = parse_entities(tg["mentioned_leaders"])
leader_counts_tw = parse_entities(tw["mentioned_leaders"])

# combine
all_leaders = sorted(
    set(leader_counts_tg) | set(leader_counts_tw),
    key=lambda x: (leader_counts_tg[x]+leader_counts_tw[x]), reverse=True
)[:15]

df_leaders = pd.DataFrame({
    "leader": all_leaders,
    "Telegram": [leader_counts_tg[l] for l in all_leaders],
    "Twitter":  [leader_counts_tw[l] for l in all_leaders],
})
df_leaders["total"] = df_leaders["Telegram"] + df_leaders["Twitter"]
df_leaders = df_leaders.sort_values("total")

fig = go.Figure()
for platform, color in [("Telegram","#2980b9"),("Twitter","#1da1f2")]:
    fig.add_trace(go.Bar(
        x=df_leaders[platform], y=df_leaders["leader"],
        name=platform, orientation="h",
        marker_color=color,
        hovertemplate="%{y}: %{x:,} mentions<extra>" + platform + "</extra>"
    ))

fig.update_layout(
    title="Top 15 Most Mentioned Leaders",
    barmode="stack", height=550,
    xaxis_title="Total Mentions",
    yaxis_title="",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    margin=dict(l=130)
)
fig.show()

### Chart 3 — Most Mentioned Geopolitical Hotspots

In [ ]:
hotspot_counts_tg = parse_entities(tg["mentioned_hotspots"])
hotspot_counts_tw = parse_entities(tw["mentioned_hotspots"])

all_hotspots = sorted(
    set(hotspot_counts_tg) | set(hotspot_counts_tw),
    key=lambda x: (hotspot_counts_tg[x]+hotspot_counts_tw[x]), reverse=True
)[:15]

df_hot = pd.DataFrame({
    "hotspot": all_hotspots,
    "Telegram": [hotspot_counts_tg[h] for h in all_hotspots],
    "Twitter":  [hotspot_counts_tw[h] for h in all_hotspots],
})
df_hot["total"] = df_hot["Telegram"] + df_hot["Twitter"]
df_hot = df_hot.sort_values("total")

fig = go.Figure()
for platform, color in [("Telegram","#8e44ad"),("Twitter","#16a085")]:
    fig.add_trace(go.Bar(
        x=df_hot[platform], y=df_hot["hotspot"],
        name=platform, orientation="h",
        marker_color=color,
        hovertemplate="%{y}: %{x:,} mentions<extra>" + platform + "</extra>"
    ))

fig.update_layout(
    title="Top 15 Most Mentioned Geopolitical Hotspots",
    barmode="stack", height=550,
    xaxis_title="Total Mentions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    margin=dict(l=120)
)
fig.show()

### Chart 4 — Sentiment Breakdown per Leader (Stacked Bar)

In [ ]:
rows = []
for df, src in [(tg,"Telegram"),(tw,"Twitter")]:
    for _, row in df.iterrows():
        for e in str(row.get("mentioned_leaders","")).split(","):
            e = e.strip()
            if e and e not in ("nan","None",""):
                rows.append({"leader":e,"sentiment":row.get("sentiment_label","neutral"),"source":src})

ls_df = pd.DataFrame(rows)

if not ls_df.empty:
    top_leaders_list = [l for l,_ in Counter(ls_df["leader"]).most_common(12)]
    ls_df = ls_df[ls_df["leader"].isin(top_leaders_list)]
    pivot = ls_df.groupby(["leader","sentiment"]).size().unstack(fill_value=0)
    pivot = pivot.reindex(
        sorted(pivot.index, key=lambda x: pivot.loc[x].sum(), reverse=True)
    )
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    fig = go.Figure()
    sent_colors = {"positive":"#27ae60","negative":"#e74c3c","neutral":"#f39c12"}
    for sent in ["positive","neutral","negative"]:
        if sent in pivot_pct.columns:
            fig.add_trace(go.Bar(
                x=pivot_pct.index,
                y=pivot_pct[sent],
                name=sent.capitalize(),
                marker_color=sent_colors[sent],
                hovertemplate="%{x}<br>" + sent + ": %{y:.1f}%<extra></extra>"
            ))

    fig.update_layout(
        title="Sentiment % per Political Leader (Top 12)",
        barmode="stack", height=500,
        yaxis_title="Share of mentions (%)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    fig.show()

### Chart 5 — Telegram Channel Performance (Views, Forwards, Reactions)

In [ ]:
ch = tg.groupby("channel_name").agg(
    posts=("message_id","count"),
    avg_views=("views","mean"),
    avg_forwards=("forwards","mean"),
    avg_reactions=("reactions_count","mean"),
    subscribers=("channel_subscribers","first")
).reset_index().round(1)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=["Avg Views per Post","Avg Forwards per Post","Avg Reactions per Post"])

for col_idx, (metric, color) in enumerate([
    ("avg_views","#2980b9"),
    ("avg_forwards","#8e44ad"),
    ("avg_reactions","#e67e22")
], 1):
    fig.add_trace(go.Bar(
        x=ch["channel_name"],
        y=ch[metric],
        marker_color=color,
        showlegend=False,
        hovertemplate="%{x}<br>" + metric + ": %{y:,.0f}<extra></extra>"
    ), row=1, col=col_idx)

fig.update_layout(
    title="Telegram Channel Comparison (engagement metrics)",
    height=420
)
fig.update_xaxes(tickangle=-25)
fig.show()

print(ch[["channel_name","posts","avg_views","avg_forwards","avg_reactions","subscribers"]].to_string(index=False))

### Chart 6 — Posts per Week + Sentiment Trend (Timeline)

In [ ]:
tg_t = tg.dropna(subset=["date"]).copy()
tg_t["week"] = tg_t["date"].dt.to_period("W").dt.start_time

weekly = tg_t.groupby("week").agg(
    posts=("message_id","count"),
    avg_sent=("sentiment_score","mean")
).reset_index()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Posts per Week (Telegram)","Average Sentiment Score per Week"])

fig.add_trace(go.Bar(
    x=weekly["week"], y=weekly["posts"],
    name="Posts", marker_color="#2980b9",
    hovertemplate="%{x|%b %d, %Y}: %{y:,} posts<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=weekly["week"], y=weekly["avg_sent"],
    name="Avg Sentiment", mode="lines+markers",
    line=dict(color="#e74c3c", width=2),
    hovertemplate="%{x|%b %d}: sentiment=%{y:.3f}<extra></extra>"
), row=2, col=1)

fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5, row=2, col=1)

fig.update_layout(title="Telegram Weekly Activity and Sentiment", height=550, showlegend=True)
fig.show()

### Chart 7 — Sentiment Score vs Word Count (Both Platforms)

In [ ]:
sample = pd.concat([
    tg[["sentiment_score","word_count","platform","channel_name"]].assign(source=tg["channel_name"]).sample(min(3000,len(tg))),
    tw[["sentiment_score","word_count","platform"]].assign(source="Twitter").sample(min(3000,len(tw)))
], ignore_index=True)

fig = px.scatter(
    sample, x="word_count", y="sentiment_score",
    color="source", opacity=0.45,
    color_discrete_sequence=px.colors.qualitative.Set1,
    hover_data=["platform"],
    labels={"word_count":"Word Count","sentiment_score":"Sentiment Score","source":"Source"},
    title="Sentiment Score vs Word Count (sample of 6,000 posts)"
)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.3)
fig.update_layout(height=480)
fig.show()

### Chart 8 — Leader × Hotspot Co-occurrence Heatmap

In [ ]:
TOP_LEADERS = [l for l,_ in parse_entities(both["mentioned_leaders"]).most_common(10)]
TOP_HOTSPOTS = [h for h,_ in parse_entities(both["mentioned_hotspots"]).most_common(10)]

matrix = pd.DataFrame(0, index=TOP_LEADERS, columns=TOP_HOTSPOTS)

for _, row in both.iterrows():
    leaders  = [x.strip() for x in str(row.get("mentioned_leaders","")).split(",") if x.strip() in TOP_LEADERS]
    hotspots = [x.strip() for x in str(row.get("mentioned_hotspots","")).split(",") if x.strip() in TOP_HOTSPOTS]
    for l in leaders:
        for h in hotspots:
            matrix.loc[l, h] += 1

fig = px.imshow(
    matrix,
    color_continuous_scale="Blues",
    aspect="auto",
    title="Leader × Hotspot Co-occurrence (darker = more posts mention both)",
    labels=dict(x="Hotspot", y="Leader", color="Co-mentions")
)
fig.update_layout(height=500)
fig.show()

### Chart 9 — Average Sentiment per Hotspot (Telegram vs Twitter)

In [ ]:
rows9 = []
for df, src in [(tg,"Telegram"),(tw,"Twitter")]:
    for _, row in df.iterrows():
        for h in str(row.get("mentioned_hotspots","")).split(","):
            h = h.strip()
            if h and h not in ("nan","None",""):
                rows9.append({"hotspot":h,"sentiment":row.get("sentiment_score",0),"platform":src})

hs_df = pd.DataFrame(rows9)
if not hs_df.empty:
    top_h = [x for x,_ in Counter(hs_df["hotspot"]).most_common(12)]
    hs_df = hs_df[hs_df["hotspot"].isin(top_h)]
    avg_sent = hs_df.groupby(["hotspot","platform"])["sentiment"].mean().reset_index()
    avg_sent.columns = ["hotspot","platform","avg_sentiment"]

    fig = px.bar(
        avg_sent, x="hotspot", y="avg_sentiment", color="platform",
        barmode="group",
        color_discrete_map={"Telegram":"#2980b9","Twitter":"#1da1f2"},
        title="Average Sentiment Score per Hotspot by Platform",
        labels={"hotspot":"Hotspot","avg_sentiment":"Avg Sentiment Score"}
    )
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig.update_layout(height=480, xaxis_tickangle=-25)
    fig.show()

### Chart 10 — Sentiment Score Distribution by Label (Violin Plot)

In [ ]:
fig = px.violin(
    both, x="sentiment_label", y="sentiment_score",
    color="platform",
    color_discrete_map={"telegram":"#2980b9","twitter":"#1da1f2"},
    box=True, points=False,
    title="Sentiment Score Distribution by Label and Platform",
    labels={"sentiment_label":"Sentiment Label","sentiment_score":"VADER Compound Score","platform":"Platform"},
    category_orders={"sentiment_label":["positive","neutral","negative"]}
)
fig.update_layout(height=480)
fig.show()

## Final Submission Summary

In [ ]:
print("=" * 60)
print("FINAL SUBMISSION SUMMARY")
print("=" * 60)

print(f"\nDataset 1 — Telegram")
print(f"  Records   : {len(tg):,}")
print(f"  Attributes: {tg.shape[1]}")
print(f"  Source    : 4 Telegram channels via Telethon API")

print(f"\nDataset 2 — Twitter/X")
print(f"  Records   : {len(tw):,}")
print(f"  Attributes: {tw.shape[1]}")
print(f"  Source    : Apify actor (keyword search)")

print(f"\nCombined  : {len(tg)+len(tw):,} records")
print(f"Min required: 10,000 records — {'PASS' if len(tg)+len(tw)>=10000 else 'FAIL'}")
print(f"Min 15 attrs: {tg.shape[1]+tw.shape[1]} total — {'PASS' if tg.shape[1]+tw.shape[1]>=15 else 'FAIL'}")
print(f"2 datasets  : PASS")
print(f"2 case studies: PASS")
print(f"3+ data sources documented: PASS")
print(f"Cleaning applied: PASS")
print(f"Visualizations: 10 Plotly charts — PASS")

print("\n" + "=" * 60)
print("Files ready for submission:")
print("  capstone1/Capstone1_Report.docx")
print("  capstone1/data/raw/telegram_raw.csv")
print("  capstone1/data/raw/twitter_raw.csv")
print("  capstone1/data/clean/telegram_raw_clean.csv")
print("  capstone1/data/clean/twitter_raw_clean.csv")
print("  capstone1/docs/step3_collection_strategy.md")
print("  capstone1/analysis.ipynb  (this file)")